In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.append('..')
sys.path.append(os.path.abspath(os.path.join('..', 'magnet-pinn')))
sys.path.append(os.path.abspath(os.path.join('..', 'neuraloperator')))

In [3]:
from torch.utils.data import DataLoader

from magnet_pinn.data.transforms import Compose, Crop, GridPhaseShift
from magnet_pinn.data.grid import MagnetGridIterator

TRAIN_DIR = "../data/processed/batch_17/grid_voxel_size_4_data_type_float32"
VAL_DIR = "../data/processed/batch_17/grid_voxel_size_4_data_type_float32"

augmentation = Compose(
    [
        Crop(crop_size=(100, 100, 100)),
        GridPhaseShift(num_coils=8)
    ]
)

val_set = MagnetGridIterator(VAL_DIR, transforms=augmentation, num_samples=8)
val_loader = iter(DataLoader(val_set, batch_size=1))

In [ ]:
import torch
from mrifield.models import AFNONet

model = AFNONet()
sample = torch.randn(1, 5, 100, 100, 100)
result = model(sample)
print(result.shape)
print(torch.norm(result))

In [ ]:
import pytorch_lightning as pl

from torch.utils.data import DataLoader

from magnet_pinn.utils import StandardNormalizer, StandardNormalizerSqrt
from magnet_pinn.data.transforms import Compose, Crop, GridPhaseShift
from magnet_pinn.data.grid import MagnetGridIterator
from magnet_pinn.data.utils import worker_init_fn

from neuralop.models import FNO, UNO
from mrifield.train.lit_mrifield import LitMRIField
from mrifield.models import AFNONet
from magnet_pinn.losses.physics import DivergenceLoss

model = FNO(n_modes=(16, 16, 16), in_channels=5, out_channels=12, hidden_channels=42)
#model = UNO(in_channels=5, out_channels=12, hidden_channels=16, n_layers=4, uno_out_channels=[32,64,64,32], uno_n_modes=[[13,13,13],[13,13,13],[13,13,13],[13,13,13]], uno_scalings=[[1,1,1],[0.5,0.5,0.5],[1,1,1],[2,2,2]], channel_mlp_skip='linear')
#model = UNO(in_channels=5, out_channels=12, hidden_channels=16, n_layers=5, uno_out_channels=[32,64,128,64,32], uno_n_modes=[[13,13,13],[13,13,13],[13,13,13],[13,13,13],[13,13,13]], uno_scalings=[[1,1,1],[0.5,0.5,0.5],[1,1,1],[1,1,1],[2,2,2]], channel_mlp_skip='linear')
#model = AFNONet()

input_normalizer = StandardNormalizer.load_from_json(f"{TRAIN_DIR}/normalization/std/input_normalization.json")
target_normalizer = StandardNormalizerSqrt.load_from_json(f"{TRAIN_DIR}/normalization/std/target_normalization.json")

augmentation = Compose(
    [
        Crop(crop_size=(100, 100, 100)),
        GridPhaseShift(num_coils=8)
    ]
)

lit_model = LitMRIField(model, input_normalizer, target_normalizer, pi_loss=DivergenceLoss())

train_set = MagnetGridIterator(TRAIN_DIR, transforms=augmentation, num_samples=100)
val_set = MagnetGridIterator(VAL_DIR, transforms=augmentation, num_samples=100)

train_loader = DataLoader(train_set, batch_size=4, num_workers=16, worker_init_fn=worker_init_fn)
val_loader = DataLoader(val_set, batch_size=4, num_workers=16, worker_init_fn=worker_init_fn)

#trainer = pl.Trainer(accelerator="cpu", devices=1, log_every_n_steps=100, max_epochs=10)
#trainer.fit(model=lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)